In [ ]:
import os
import math
import numpy as np
import pandas as pd
from typing import Literal, Tuple
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from src.datasets.single_gal_sims import sim_simple_gal
import matplotlib.pyplot as plt

import anacal
from src.architecture.single_gal_CNN import R180Inv_CNN_GeLU, D4T_CNN_GeLU
from src.architecture.CNN_toolkit import shape_pixel_gradients, predictor, plot_shape_bidirectional, D4_eq_weight
from src.training import train_model
from src.anacal.pixel_response import anacal_pix_r

device = ("cuda" if torch.cuda.is_available() else "cpu")
#base = R180Inv_CNN_GeLU().to(device)
model = D4T_CNN_GeLU().to(device)
model.load_state_dict(torch.load("./models/D4T_CNN_nl_10ep.pth", map_location="cpu"))
_ = model.eval()

In [ ]:
gal_img, psf_img = sim_simple_gal(0, 0, 0,0, psf_model = 'Gaussian',return_psf = True)
gal_img_p, psf_img_p = sim_simple_gal(0.6, 0.4, 0.01,0, psf_model = 'Moffat', psf_para = [2.5, 0.7],return_psf = True)

q_img_np = anacal_pix_r(gal_img, psf_img, sigma_arcsec=0.7/2.355)
q_img_np_p = anacal_pix_r(gal_img_p, psf_img_p, sigma_arcsec=0.7/2.355)

In [ ]:
ngrid = 32
g1 = 0.03; g2 = 0.00
obj = galsim.Sersic(n=1.3, half_light_radius=1.5, flux =20.0).shear(g1=g1, g2=g2)
psf = galsim.Gaussian(sigma=2.0); obj = galsim.Convolve(obj, psf)
im = obj.shift(0.5,0.5).drawImage(nx=ngrid, ny=ngrid, scale=1, method = 'sb').array.astype(np.float64)
pixel_response = get_pixel_response(im, sigma=2.0)

In [ ]:
#model = D4T_CNN_GeLU().to(device)
model = train_model(
    model,
    images_path="/projects/bdsp/wenyinli/datasets/simple_gal_images.npy",
    csv_path="/projects/bdsp/wenyinli/datasets/simple_gal_info.csv",
    target="e",          # or "g" if you want to train on shear labels
    epochs=10,
    batch_size=512,
    num_workers=32,
    lr=1e-3,
)
torch.save(model.state_dict(), './D4T_CNN_nl_10ep.pth')

In [ ]:
img, y_gt = load_test_item(images_path="/projects/bdsp/wenyinli/datasets/simple_gal_images.npy",
                           csv_path="/projects/bdsp/wenyinli/datasets/simple_gal_info.csv", 
                           index=11, target="e")

y_pred = predictor(model, img) 

plot_shape_bidirectional(img, y_gt, y_pred=y_pred, arrow_scale=0.4, fixed_length=False)

In [ ]:
image_p, psf_p = sim_simple_gal(0.4, 0.1, 0.0,0,sersic_n = 1,image_size=64, shift = [0,0], gal_flux = 500,
                         psf_model = 'Moffat', psf_para = [2.5, 0.7],return_psf = True, noise_std = 0.0)
image_n, psf_n = sim_simple_gal(-0.4, -0.1, 0.0,0,sersic_n = 1,image_size=64, shift = [1,1],gal_flux = 500,
                         psf_model = 'Moffat', psf_para = [2.5, 0.7],return_psf = True, noise_std = 0.0)
noise, noise_psf = sim_simple_gal(0, 0, 0.0,0,sersic_n = 1,image_size=64, shift = [0,0], gal_flux = 0,
                         psf_model = 'Moffat', psf_para = [2.5, 0.7],return_psf = True, noise_std = 0.05)

q_p = anacal_pix_r(image_p, psf_p, noise_map = noise)
q_n = anacal_pix_r(image_n, psf_n, noise_map = noise)
p_pred = predictor(model, q_p[0].astype(np.float32),normalize="None")
n_pred = predictor(model, q_n[0].astype(np.float32),normalize="None")
print(p_pred)
print(n_pred)
#print((p_pred+n_pred))

In [ ]:
import time
start_time = time.time()
pred, grad_e1, grad_e2 = shape_pixel_gradients(model, q_n[0].astype(np.float32), normalize="none")
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Elapsed time: {elapsed_time:.4f} seconds")

In [ ]:
features = {}

# 定义 hook 函数
def get_features(name):
    def hook(model, input, output):
        features[name] = output.detach()
    return hook

# 给想要观察的层注册 hook
model.blocks[4].register_forward_hook(get_features("conv1"))
_ = predictor(model, q_n[0].astype(np.float32),normalize="None")


In [ ]:
#plt.imshow(features['conv1'][0][11].cpu().numpy(), cmap="bwr", origin="lower")
plt.imshow(model.w1[0,0].detach().cpu().numpy(), cmap="bwr", origin="lower")
#plt.imshow(model.y[0,7].detach().cpu().numpy(), cmap="bwr", origin="lower")
plt.colorbar()
plt.show()

In [ ]:
#q_p,q_n, q_m, image_p, image_n, image_m = mid_prod

#img = image_p
def draw_grad(model, img):
    pred, grad_e1, grad_e2 = shape_pixel_gradients(model, img, normalize="none")

    print("pred (e1,e2):", pred)


    # 可视化
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 3, figsize=(10, 3))
    im0 = axs[0].imshow(np.log(np.abs(img)), cmap="gray", origin="lower")
    axs[0].set_title("input")
    im1 = axs[1].imshow(grad_e1[0][0], cmap="bwr", origin="lower", vmax = np.abs(grad_e1).max(), vmin = -np.abs(grad_e1).max())
    axs[1].set_title(r"$\partial e_1/\partial I$")
    im2 = axs[2].imshow(grad_e2[0][0], cmap="bwr", origin="lower", vmax = np.abs(grad_e2).max(), vmin = -np.abs(grad_e2).max())
    axs[2].set_title(r"$\partial e_2/\partial I$")

    fig.colorbar(im0, ax=axs[0])
    fig.colorbar(im1, ax=axs[1])
    fig.colorbar(im2, ax=axs[2])

    for ax in axs:
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()
    return pred, grad_e1, grad_e2

pred, grad_e1, grad_e2 = draw_grad(model, q_p[0].astype(np.float32))


In [ ]:
import numpy as np
import torch

N = 100
def simple_bias(model, shear=[1e-2, 0], e1 = 0, e2 = 0, gal_flux=1000, sersic_n = 1, gal_hlr = 0.5):
    common_kwargs = dict(e1=e1, e2=e2,  
                         gal_half_light_radius=gal_hlr, 
                         sersic_n=sersic_n,
                         psf_model = 'Moffat', 
                         psf_para = [2.5, 0.7],
                         return_psf = True,
                         pixel_scale=0.2, 
                         image_size=64,
                         shift = [0,0], 
                         noise_std=0.1, 
                         show=False)
    noise, psf_noise = sim_simple_gal(g1=0, g2=0, gal_flux=0,seed = 888,**common_kwargs)
    image_p, psf_p = sim_simple_gal(g1=+shear[0], g2=+shear[1],gal_flux=gal_flux,seed = 666, **common_kwargs)
    image_n, psf_n = sim_simple_gal(g1=-shear[0], g2=-shear[1], gal_flux=gal_flux,seed = 666,**common_kwargs)
    image_m, psf_m = sim_simple_gal(g1=0, g2=0,gal_flux=gal_flux,seed = 666, **common_kwargs)

    q_m = anacal_pix_r(image_m, psf_m, noise_map = noise)
    q_p = anacal_pix_r(image_p, psf_p, noise_map = noise)
    q_n = anacal_pix_r(image_n, psf_n, noise_map = noise)

    p_pred = predictor(model, q_p[0].astype(np.float32), normalize="none")
    n_pred = predictor(model, q_n[0].astype(np.float32), normalize="none")
    m_pred, grad_e1, grad_e2 = shape_pixel_gradients(model, q_m[0].astype(np.float32), normalize="none")
    grad_e1 = grad_e1[0][0]
    grad_e2 = grad_e2[0][0]
    delta_img = (q_p[0] - q_n[0])
    
    '''p_pred = predictor(model, image_p, normalize="none")
    n_pred = predictor(model, image_n, normalize="none")
    m_pred, grad_e1, grad_e2 = shape_pixel_gradients(model, image_m, normalize="none")
    delta_img = (image_p - image_n)'''
    
    
    lin_e    = [(grad_e1 * delta_img).sum(), (grad_e2 * delta_img).sum()]
    fd_e     = (p_pred - n_pred)
    response = [[((q_m[1])*grad_e1).sum(), (q_m[2]*grad_e1).sum()],
                [(q_m[1]*grad_e2).sum(), (grad_e2*q_m[2]).sum()]]
    return lin_e, fd_e, response, [grad_e1, grad_e2, m_pred], [q_p,q_n, q_m, image_p, image_n, image_m]

def calibration_result(e_list, r_list):
    e = np.mean(e_list, axis = 0) # [2]
    r = np.mean(r_list, axis = 0) # [2,2]
    return np.array([e[0]/r[0,0], e[1]/r[1,1]])

truth_paras = []
lin_e_list = []
fd_e_list = []
r_list = []
pred_e_list = []
shear = [1e-2, 0]
for gal_index in range(0,N):
    e1 = np.random.uniform(-0.6, 0.6)
    e2 = np.random.uniform(-0.6, 0.6)
    gal_flux = 10**np.random.uniform(2.5, 3.5)
    sersic_n = np.random.uniform(1, 5)
    gal_hlr = np.exp(np.random.uniform(np.log(0.6), np.log(1.2)))
    #print([e1, e2, gal_flux, sersic_n, gal_hlr])
    lin_e, fd_e, response, grads,mid_prod = simple_bias(model,shear = shear, 
                                                          e1 = e1, e2 =e2, 
                                                          gal_flux=gal_flux, sersic_n=sersic_n, gal_hlr=gal_hlr)
    lin_e_list.append(lin_e)
    fd_e_list.append(fd_e)
    r_list.append(response)
    pred_e_list.append(grads[2]) 
    truth_paras.append([e1, e2, gal_flux, sersic_n, gal_hlr])
    if (gal_index+1)%200 == 0:
        print("N_sample = ", gal_index+1)
        ratio_e  =  np.mean(fd_e_list,axis = 0)/np.mean(lin_e_list, axis = 0)*0.02
        e_r = calibration_result(fd_e_list, r_list)
        print("Delta_img bias =", ratio_e/0.02-1)
        print("m1, m2 =", e_r/0.02-1)

print("##########################################")
print("N_sample = ", gal_index+1)
ratio_e  =  np.mean(fd_e_list,axis = 0)/np.mean(lin_e_list, axis = 0)*0.02
e_r = calibration_result(fd_e_list, r_list)
c = np.mean(fd_e_list,axis = 0)
truth_paras = np.array(truth_paras)
print("Delta_img bias =", ratio_e/0.02-1)
print("m1, m2 =", e_r/0.02-1)



In [ ]:
plt.scatter(np.array(fd_e_list)[:,0], np.array(lin_e_list)[:,0], s = 5)
plt.xlabel('Model output Shear')
plt.ylabel('model_grad*delta_img')
plt.show()

plt.scatter(np.array(truth_paras)[:,0], np.array(pred_e_list)[:,0,0], c=np.log10(truth_paras[:,2]),  s = 5)
plt.colorbar(label = 'log10 of total flux')
plt.xlabel('Truth e1')
plt.ylabel('Pred e1')
plt.show()

In [ ]:
plt.close()
q_p,q_n, q_m, image_p, image_n, image_m, q_noise = mid_prod

plt.imshow(q_p[1])
plt.colorbar()
plt.show()


plt.imshow(grads[0]*(q_p[0]-q_n[0]-q_m[1]*0.02))
print((grads[0]*(q_p[0]-q_n[0]-q_m[1]*0.02)).sum())
plt.colorbar()
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(15, 10))
q_p,q_n, q_m, image_p, image_n, image_m = mid_prod
# Plot pixel response
im0 = axs[0, 0].imshow(q_m[1]*0.02, cmap="bwr", origin="lower")
axs[0, 0].set_title('pixel response *0.02')
fig.colorbar(im0, ax=axs[0, 0])

# Print sum of pixel response
print((q_m[1]*0.02*(q_p[0]-q_n[0])).sum())

# Plot difference image
im1 = axs[0, 1].imshow((q_p[0]-q_n[0]), cmap="bwr", origin="lower")
axs[0, 1].set_title('Difference image')
fig.colorbar(im1, ax=axs[0, 1])
print((mid_prod[2]*mid_prod[0]).sum())

# Plot Grad_e1
im2 = axs[0, 2].imshow(q_m[1], cmap="bwr", origin="lower")
axs[0, 2].set_title('Grad_e1')
fig.colorbar(im2, ax=axs[0, 2])

# Plot Res_img = (delta-response*0.02)
im3 = axs[1, 0].imshow(((q_p[0]-q_n[0])-q_m[1]*0.02), cmap="bwr", origin="lower")
axs[1, 0].set_title('Res_img = (delta-response*0.02)')
fig.colorbar(im3, ax=axs[1, 0])

# Plot Res_img = (delta-response*0.02)*grad_e1
im4 = axs[1, 1].imshow(((q_p[0]-q_n[0])-q_m[1]*0.02)*grads[0], cmap="bwr", origin="lower")
axs[1, 1].set_title('Res_img = (delta-response*0.02)*grad_e1')
fig.colorbar(im4, ax=axs[1, 1])

# Hide the last subplot (optional)
# Plot Res_img = (delta-response*0.02)*grad_e1
im5 = axs[1, 2].imshow(((q_p[0]-q_n[0]))*grads[0], cmap="bwr", origin="lower")
axs[1, 2].set_title('(response*0.02)*grad_e1')
fig.colorbar(im5, ax=axs[1, 2])

plt.tight_layout()
plt.show()


N_sample =  200
Delta_img bias = 0.0010575877104046505
m1 = 0.0005826022598740543
N_sample =  400
Delta_img bias = 0.0006978188445256883
m1 = 0.00017243663677168897
N_sample =  600
Delta_img bias = 6.578559398651151e-05
m1 = -0.0004351105601261285
N_sample =  800
Delta_img bias = 6.405366975248405e-05
m1 = -0.00044608869118434313
N_sample =  1000
Delta_img bias = -0.00024758815154002356
m1 = -0.0007353604069224584
##########################################
Final Delta_img bias = -0.00024758815154002356
Final m1 = -0.0007353604069224584